# 98 — Dashboard export

Write `dashboard_data.parquet` for the React dashboard. Layout: one row per land grid cell, with

- underscore-prefixed metadata columns (`_lat`, `_lon`, `_country_code`, `_continent`) that the dashboard treats as non-scoring context,
- every percentile-normalized layer from `normalized.nc` **except** `temperature_pleasantness`, `precipitation_balance`, and `population_density` — those three are replaced by profile-based variants,
- for each of the three replaced layers, one column per profile in `TEMP_PROFILES` / `PRECIP_PROFILES` / `DENSITY_PROFILES`, suffixed `_p1`, `_p2`, …, and percentile-normalized so they share the same `[0, 1]` uniform distribution as the other layers.

In [1]:
import numpy as np
import pandas as pd
import xarray as xr
from scipy.stats import rankdata

from common import (
    DENSITY_PROFILES,
    PRECIP_PROFILES,
    PROCESSED_DIR,
    TEMP_PROFILES,
    compute_population_density_score,
    compute_precipitation_balance,
    compute_temperature_pleasantness,
    load_raw_scoring_inputs,
)

## Profiles

Imported from `common.py` so this notebook and `99_atlas_map.ipynb` stay in sync. `(ideal, tolerance)` for temperature and precipitation; `((min, max), tolerance_decades)` for density where `None` means unbounded on that side — wilderness is a pure upper bound (empty cells still score 1), urban a pure lower bound.

## Helpers

`percentile_rank` mirrors the transform in `92_normalization.ipynb` so the profile columns share the same uniform `[0, 1]` distribution as the base layers already in `normalized.nc`.

In [2]:
def percentile_rank(da):
    """Rank-transform a DataArray to percentiles in ``[0, 1]``.

    Ties are averaged. NaN cells are preserved and excluded from the ranking.
    Shares dims, coords and name with the input so it can be dropped into any
    consumer that previously took the raw layer.
    """
    values = da.values.astype(float)
    flat = values.ravel()
    mask = np.isfinite(flat)
    out = np.full_like(flat, np.nan)
    n = int(mask.sum())
    if n > 1:
        out[mask] = (rankdata(flat[mask], method='average') - 1) / (n - 1)
    elif n == 1:
        out[mask] = 0.0
    return da.copy(data=out.reshape(values.shape))

## Base layers

Start from `normalized.nc` (percentile-transformed by `92_normalization.ipynb`) and drop the three layers we're replacing with profile-based variants.

In [3]:
REPLACED = ('temperature_pleasantness', 'precipitation_balance', 'population_density')

ds = xr.open_dataset(PROCESSED_DIR / 'normalized.nc')
base = ds.drop_vars([v for v in REPLACED if v in ds.data_vars])
list(base.data_vars)

['sea_proximity',
 'terrain_ruggedness',
 'sun_hours',
 'annual_greenness',
 'climate_vulnerability',
 'natural_disaster_risk',
 'air_quality',
 'urbanity',
 'internet_connectivity',
 'healthcare_access',
 'income',
 'cost_of_living',
 'crime_rate',
 'human_freedom',
 'corruption']

## Profile layers

For each profile, compute the raw comfort score with the helper in `common.py`, then percentile-rank so the result is comparable to the other columns. Column names are `<base>_p1`, `_p2`, … in dict-iteration order — the human-readable labels stay in this notebook (and are mirrored in the dashboard).

In [4]:
temp_da, precip_da, density_da = load_raw_scoring_inputs()

profile_layers = {}

for i, (label, (ideal, tol)) in enumerate(TEMP_PROFILES.items(), start=1):
    score = compute_temperature_pleasantness(temp_da, ideal_temp=ideal, tolerance=tol)
    profile_layers[f'temperature_pleasantness_p{i}'] = percentile_rank(score)
    print(f'temperature_pleasantness_p{i}  <- {label}')

for i, (label, (ideal, tol)) in enumerate(PRECIP_PROFILES.items(), start=1):
    score = compute_precipitation_balance(precip_da, ideal_monthly_mm=ideal, tolerance_mm=tol)
    profile_layers[f'precipitation_balance_p{i}'] = percentile_rank(score)
    print(f'precipitation_balance_p{i}     <- {label}')

for i, (label, (rng, tol_dec)) in enumerate(DENSITY_PROFILES.items(), start=1):
    score = compute_population_density_score(density_da, density_range=rng, tolerance_decades=tol_dec)
    profile_layers[f'population_density_p{i}'] = percentile_rank(score)
    print(f'population_density_p{i}        <- {label}')

temperature_pleasantness_p1  <- Icy (5°C ±8)
temperature_pleasantness_p2  <- Cool (15°C ±8)
temperature_pleasantness_p3  <- Temperate (20°C ±10)
temperature_pleasantness_p4  <- Warm (25°C ±8)
temperature_pleasantness_p5  <- Tropical (27°C ±5)
precipitation_balance_p1     <- Arid (20mm ±20)
precipitation_balance_p2     <- Balanced (80mm ±60)
precipitation_balance_p3     <- Wet (150mm ±60)
precipitation_balance_p4     <- Very wet (250mm ±80)
population_density_p1        <- Wilderness (≤1/km² ±1dec)
population_density_p2        <- Rural (10–300/km² ±1dec)
population_density_p3        <- Suburban (200–1500/km² ±1dec)
population_density_p4        <- Urban (≥1500/km² ±1dec)


## Merge and flatten

Combine the base and profile layers on the shared `(lat, lon)` grid, flatten to one row per cell, and drop cells that are NaN across every value column (pure ocean / coverage gaps) before the spatial join.

In [5]:
combined = xr.Dataset({**base.data_vars, **profile_layers})
df = combined.to_dataframe().reset_index()

value_cols = [c for c in df.columns if c not in ('lat', 'lon')]
df = df.dropna(subset=value_cols, how='all').reset_index(drop=True)
len(df)

68294

## Country & continent

Look up each cell's country from the cached `03_region_mask.ipynb` mask (dominant-coverage assignment, so island cells and coastal cells whose centre sits in the ocean still get a country). Continent comes from the accompanying `load_country_lookup()` table. Ocean / disputed cells stay NaN in both columns.

In [6]:
from common import load_country_lookup, load_country_mask

iso_per_cell = load_country_mask()
mask_df = (
    iso_per_cell.to_dataframe(name='country_code')
    .reset_index()
    .replace({'country_code': ''}, {'country_code': np.nan})
)

continent_map = dict(zip(load_country_lookup()['ISO3'], load_country_lookup()['CONTINENT']))
mask_df['continent'] = mask_df['country_code'].map(continent_map)

df = df.merge(mask_df[['lat', 'lon', 'country_code', 'continent']], on=['lat', 'lon'], how='left')

n_missing = df['country_code'].isna().sum()
print(f'{len(df):,} cells; {n_missing:,} ({n_missing / len(df):.1%}) with no country match')

68,294 cells; 805 (1.2%) with no country match


## Save

Rename metadata columns to the underscore-prefixed form the dashboard expects (`_lat`, `_lon`, `_country_code`, `_continent`) and write.

In [7]:
df = df.rename(columns={
    'lat': '_lat',
    'lon': '_lon',
    'country_code': '_country_code',
    'continent': '_continent',
})

out = PROCESSED_DIR / 'dashboard_data.parquet'
df.to_parquet(out, index=False)
print(f'wrote {out} ({out.stat().st_size / 1024**2:.1f} MB, {len(df):,} rows, {len(df.columns)} cols)')
df.head()

wrote /Users/lutz/Documents/alfatraining/projects/data-science-notebooks/data/world-livable-atlas/processed/dashboard_data.parquet (7.6 MB, 68,294 rows, 32 cols)


,_lat,_lon,sea_proximity,terrain_ruggedness,sun_hours,annual_greenness,climate_vulnerability,natural_disaster_risk,air_quality,urbanity,...,precipitation_balance_p1,precipitation_balance_p2,precipitation_balance_p3,precipitation_balance_p4,population_density_p1,population_density_p2,population_density_p3,population_density_p4,_country_code,_continent
0,-76.25,168.25,0.868435,0.811597,0.108469,NaN,NaN,NaN,NaN,0.170164,...,0.974734,0.127503,0.230245,0.386925,0.72719,0.273088,0.386204,0.465546,NaN,NaN
1,-59.25,-27.25,0.853030,0.984490,0.107956,NaN,NaN,NaN,NaN,0.170164,...,0.084887,0.998389,0.482951,0.386925,0.72719,0.273088,0.386204,0.465546,NaN,NaN
2,-59.25,-26.75,0.843073,0.958699,0.107956,NaN,NaN,NaN,NaN,0.170164,...,0.084887,0.998081,0.484870,0.386925,0.72719,0.273088,0.386204,0.465546,NaN,NaN
3,-58.25,-26.25,0.874365,0.989250,0.057926,NaN,NaN,NaN,NaN,0.170164,...,0.084887,0.999883,0.517276,0.386925,0.72719,0.273088,0.386204,0.465546,SGS,Seven seas (open ocean)
4,-57.75,-26.25,0.906301,0.980184,0.050616,NaN,NaN,NaN,NaN,0.170164,...,0.084887,0.999927,0.517899,0.386925,0.72719,0.273088,0.386204,0.465546,NaN,NaN


## Grid-cell polygons

Export one `Polygon` feature per row of `dashboard_data.parquet` as raw `[lon, lat]` corner coordinates (RFC 7946, WGS84). The dashboard reprojects them at render time — see `geometry.ts::transformCoordinates` / `buildCell`, which sample intermediate points along each edge before projecting so cells stay curved on the map.

Features are in parquet-row order; the dashboard joins each polygon to its scalar value by array index.

In [8]:
# Half of the grid cell size in degrees (matches RES = 0.5 in the pipeline;
# cells are 0.5° squares centered on _lat / _lon).
HALF = 0.25


def build_cell(lon, lat):
    """Return a closed lon/lat ring (5 points, counter-clockwise) for a grid
    cell centered at (lon, lat).

    Coordinates are rounded to 2 decimal places — grid centers land on the
    0.5° grid so 2 dp is exact, and rounding also shields the emitted JSON
    from float-repr noise like ``45.24999999999``.
    """
    w = round(lon - HALF, 2)
    e = round(lon + HALF, 2)
    s = round(lat - HALF, 2)
    n = round(lat + HALF, 2)
    return [[w, s], [e, s], [e, n], [w, n], [w, s]]

In [9]:
import json

# Feature order matches parquet row order — the dashboard joins by array index,
# so we don't duplicate the index into properties.
features = [
    {
        'type': 'Feature',
        'properties': {},
        'geometry': {
            'type': 'Polygon',
            'coordinates': [build_cell(lon, lat)],
        },
    }
    for lon, lat in zip(df['_lon'].to_numpy(), df['_lat'].to_numpy())
]

fc = {'type': 'FeatureCollection', 'features': features}

geo_out = PROCESSED_DIR / 'dashboard_cells.geojson'
geo_out.write_text(json.dumps(fc, separators=(',', ':')))
print(f'wrote {geo_out} ({geo_out.stat().st_size / 1024**2:.1f} MB, {len(features):,} features)')

wrote /Users/lutz/Documents/alfatraining/projects/data-science-notebooks/data/world-livable-atlas/processed/dashboard_cells.geojson (9.6 MB, 68,294 features)
